# Notebook 36 — Chronological thresholds and constrained portfolio simulation

**Question:** Does the OOF meta-logistic ranking advantage persist when entry thresholds are fixed using earlier calibration observations, with per-day position caps, turnover costs and optional sector exposure limits?

**Inputs:** Notebook 34 saved development-validation predictions and its training-period-calibrated policy thresholds. Notebook 35 retrospective top-k rankings are **not** used for trade selection.

**Scope:** Diagnostic next-bar equal-weight simulation, not a broker-ready backtest. Assumes complete one-bar positions and frictionless entry/exit except stated costs. Does not access the closed final holdout or prospective ledger.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
if not (ROOT/'src').exists():raise FileNotFoundError('Run inside stock_ml_dynamic repository')
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd,numpy as np,matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.deployable_policy_research import run_research,load_inputs
PRED=ROOT/'artifacts/notebook34_calibration_coverage/validation_predictions.parquet'
POLICIES=ROOT/'artifacts/notebook34_calibration_coverage/policies.csv'
OUTPUT=ROOT/'artifacts/deployable_policy/1d'
print('Predictions:',PRED,'exists:',PRED.exists())
print('Prior-calibration policies:',POLICIES,'exists:',POLICIES.exists())
if not PRED.exists() or not POLICIES.exists():
    raise FileNotFoundError('Run Notebook 34 first; check its OUTPUT path for predictions and policies.')

## 1. Verify fixed, earlier-period thresholds

Notebook 34 derives these thresholds from calibration observations preceding each outer validation fold. This notebook **does not** optimize them using validation labels, returns, or score quantiles. The three models are evaluated over the same validation days.

In [ ]:
COVERAGE=.10
CALIBRATION='raw'
pred,thresholds=load_inputs(PRED,POLICIES,calibration=CALIBRATION,coverage=COVERAGE)
display(thresholds[['fold','model','target_coverage','threshold']].sort_values(['fold','model']))
print('Validation observations:',len(pred),'folds:',sorted(pred.fold.unique()))
print('Validation date range:',pred.bar_timestamp.min(),pred.bar_timestamp.max())

## 2. Portfolio assumptions

- Select stocks only when their score exceeds the **fixed earlier-calibrated threshold**.
- Rank eligible names by current score; take at most **10 positions per date** (a cross-sectional cap, not a persistent simultaneous-position engine).
- Each position is held for **one next-bar interval**; capital is divided into 10 equal slots, with unused slots held as cash.
- Deduct **10 bps per selected position** plus a **5 bps new-name turnover proxy**. This does not model spread dynamics, market impact, fills, dividends, or financing.
- Optional sector cap requires an explicit complete, point-in-time `symbol,sector` CSV. Without one, sector constraints are **not** claimed.
- `COOLDOWN_DAYS=0` allows repeat symbols on consecutive dates. Increase for a deliberate cooldown sensitivity check.

In [ ]:
MAX_POSITIONS=10
MAX_PER_SECTOR=None
SECTOR_CSV=None  # Set to an audited point-in-time CSV with columns symbol,sector before enabling a sector cap
COOLDOWN_DAYS=0
COST_BPS=10
TURNOVER_BPS=5
result=run_research(PRED,POLICIES,coverage=COVERAGE,calibration=CALIBRATION,
    max_positions=MAX_POSITIONS,max_per_sector=MAX_PER_SECTOR,sector_csv=SECTOR_CSV,
    cooldown_days=COOLDOWN_DAYS,cost_bps=COST_BPS,turnover_bps=TURNOVER_BPS,
    reps=1000,block_days=10,output_dir=OUTPUT)
scores=result['scores'];daily=result['daily'];trades=result['trades'];confidence=result['confidence']
display(scores.round(6))

## 3. Fold-by-fold portfolio comparison

Report coverage **actually achieved**, average fraction of portfolio slots filled, daily equal-weight net return, compounded historical return, and drawdown. The compounding is a simplified next-bar illustration and should not be interpreted as executable performance.

In [ ]:
cols=['fold','model','days','entries','coverage','up_precision','avg_invested_fraction','mean_net_per_trade','mean_daily_net','total_compounded_net','max_drawdown','mean_daily_new_names']
display(scores[cols].sort_values(['fold','model']).round(6))
summary=scores.groupby('model')[['coverage','up_precision','avg_invested_fraction','mean_daily_net','max_drawdown']].mean()
display(summary.round(6))
fig,ax=plt.subplots(figsize=(9,4))
for model,g in daily.groupby('model'):
    bydate=g.groupby('bar_timestamp').net_portfolio_return.mean().sort_index()
    ax.plot(bydate.index,(1+bydate).cumprod()-1,label=model)
ax.set_title('Illustrative compounded daily portfolio returns (folds averaged on overlapping dates)')
ax.set_ylabel('Compounded net return');ax.legend();plt.xticks(rotation=30);plt.tight_layout();plt.show()

## 4. Paired uncertainty across market dates

Moving-block bootstrap (10-day blocks) of **paired daily portfolio return differences** for each chronological fold. These intervals describe historical sampling variability, not uncertainty from retraining models or live execution.

In [ ]:
display(confidence.round(7))
print('Confidence intervals excluding zero:',int(((confidence.ci_low>0)|(confidence.ci_high<0)).sum()),'of',len(confidence))

## 5. Threshold and capacity sensitivity

Coverage levels are **pre-specified** (5%, 10%, 15%, 20%, 25%) and their thresholds were already calibrated in Notebook 34. Evaluate 5 and 10 positions per day without selecting a winner based on this same validation set. Treat all comparisons as exploratory.

In [ ]:
sensitivity=[]
for cov in [.05,.10,.15,.20,.25]:
    for cap in [5,10]:
        r=run_research(PRED,POLICIES,coverage=cov,calibration=CALIBRATION,
            max_positions=cap,cost_bps=COST_BPS,turnover_bps=TURNOVER_BPS,
            reps=20,block_days=10,output_dir=None)
        tab=r['scores'].copy();tab['target_coverage']=cov;tab['max_positions']=cap
        sensitivity.append(tab)
sensitivity=pd.concat(sensitivity,ignore_index=True)
display(sensitivity.groupby(['target_coverage','max_positions','model'])[['coverage','up_precision','mean_daily_net','avg_invested_fraction']].mean().round(6))
OUTPUT.mkdir(parents=True,exist_ok=True)
sensitivity.to_csv(OUTPUT/'sensitivity.csv',index=False)

## 6. Research interpretation and limitations

- Compare the **same fold/date pairs**, not just aggregate precision.
- Check whether a model's apparent edge persists after cash drag, capacity constraints, and churn costs.
- A score threshold calibrated on earlier data can still drift badly in later regimes; compare achieved coverage with target coverage.
- Sector exposure limits are **not implemented** unless an audited point-in-time sector map is supplied. Never substitute current sector membership for historical membership without disclosure.
- These are next-bar, daily grouped stock observations, not actual broker fills or an event-driven portfolio simulator. Overlapping return horizons and intra-day execution are not modeled.
- Do not tune the final strategy on the closed historical holdout. The prospective ledger is unchanged.

In [ ]:
print('Notebook 36 outputs:',OUTPUT)
print('Saved daily portfolio returns, selected trades, confidence intervals and sensitivity tables.')
print('Closed final holdout and prospective ledger were not accessed.')